# Milestone 2: Principal Component Analysis – Anderson Cancer Center Referrals
Pipeline: load with an explicit schema → clean → assemble & standardize → PCA (2 components)
→ identify essential variables → (bonus) logistic regression.

## 1. Start Spark and define the schema

In [47]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

DATA_FILE = "BAN6420 M5 breast_cancer_referrals.csv"
TEXT_COLUMNS = ["patient_id", "diagnosis"]      # every other column is a numeric measurement

# Start the Spark engine (hide routine log messages)
spark = SparkSession.builder.appName("BreastCancerPCA").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Read only the header row to get the 32 column names, then build the schema:
# text columns -> StringType, measurements -> DoubleType
all_columns = pd.read_csv(DATA_FILE, nrows=0).columns.tolist()
schema = StructType([
    StructField(col, StringType() if col in TEXT_COLUMNS else DoubleType(), True)
    for col in all_columns
])

## 2. Load the data and check its quality

In [48]:
raw_df = spark.read.csv(DATA_FILE, header=True, schema=schema)
print("Rows:", raw_df.count(), "| Columns:", len(raw_df.columns))

# here am showing the issues in detail
duplicates = raw_df.count() - raw_df.dropDuplicates().count()
print("Duplicate rows:", duplicates)

for col in all_columns:
    missing = raw_df.filter(raw_df[col].isNull()).count()
    if missing > 0:
        print(f"Missing values in {col}: {missing}")

Rows: 572 | Columns: 32
Duplicate rows: 3
Missing values in mean_radius: 6
Missing values in mean_texture: 6
Missing values in worst_area: 6


## 3. Clean the data
Duplicates are removed first so they don't affect anything else. Only 18 rows (about 3%)
have missing values, so they are dropped rather than filled; this keeps only real
measurements and still leaves 551 patients for analysis.

In [49]:
clean_df = raw_df.dropDuplicates().dropna()
print("Rows before cleaning:", raw_df.count())
print("Rows after cleaning: ", clean_df.count())

Rows before cleaning: 572
Rows after cleaning:  551


## 4. Assemble and standardize the features
`patient_id` (an identifier) and `diagnosis` (the outcome) are excluded from the features.
The 30 measurements are packed into one vector, then standardized to mean 0 and
standard deviation 1 so that large-unit variables (e.g. area) don't dominate small ones (e.g. smoothness).

In [50]:
feature_columns = [col for col in clean_df.columns if col not in TEXT_COLUMNS]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features")
assembled_df = assembler.transform(clean_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features",
                        withMean=True, withStd=True)
prepared_df = scaler.fit(assembled_df).transform(assembled_df)

prepared_df.select("patient_id", "raw_features", "scaled_features").show(3)

+----------+--------------------+--------------------+
|patient_id|        raw_features|     scaled_features|
+----------+--------------------+--------------------+
|  PT-00001|[17.99,10.38,122....|[1.08676155287363...|
|  PT-00046|[18.65,17.6,123.7...|[1.27284502443201...|
|  PT-00500|[20.59,21.24,137....|[1.81981765295211...|
+----------+--------------------+--------------------+
only showing top 3 rows


## 5. PCA: reduce 30 features to 2 principal components

In [51]:
pca = PCA(k=2, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(prepared_df)
pca_df = pca_model.transform(prepared_df)

# Explained variance: how much of the information each component keeps
variance = pca_model.explainedVariance
print(f"PC1 explains: {variance[0] * 100:.2f}%")
print(f"PC2 explains: {variance[1] * 100:.2f}%")
print(f"Total kept:   {sum(variance) * 100:.2f}%")

pca_df.select("patient_id", "pca_features").show(3, truncate=False)

PC1 explains: 44.22%
PC2 explains: 19.04%
Total kept:   63.27%
+----------+----------------------------------------+
|patient_id|pca_features                            |
+----------+----------------------------------------+
|PT-00001  |[-9.126366945143804,-1.9700125520752754]|
|PT-00046  |[-5.341679842730322,0.11624207720207567]|
|PT-00500  |[-5.050027040833921,1.9966552955200398] |
+----------+----------------------------------------+
only showing top 3 rows


## 6. Essential variables: which measurements drive each component?
Each component is a weighted combination of the 30 variables. The largest weights
(loadings) show the most influential variables.

In [52]:
# pc = the weights (30 variables x 2 components); abs() because large negative weights matter too
loadings_df = pd.DataFrame(pca_model.pc.toArray(), index=feature_columns, columns=["PC1", "PC2"])

for pc in ["PC1", "PC2"]:
    print(f"\nTop 5 variables in {pc}:")
    print(loadings_df[pc].abs().sort_values(ascending=False).head(5).round(3))


Top 5 variables in PC1:
mean_concave_points     0.261
mean_concavity          0.258
worst_concave_points    0.251
mean_compactness        0.239
worst_perimeter         0.237
Name: PC1, dtype: float64

Top 5 variables in PC2:
mean_fractal_dimension     0.367
fractal_dimension_error    0.282
worst_fractal_dimension    0.276
compactness_error          0.234
mean_radius                0.233
Name: PC2, dtype: float64


## 7. Bonus: logistic regression on the 2 components

In [53]:
# Convert diagnosis text to numbers: B (most common) = 0, M = 1
indexer = StringIndexer(inputCol="diagnosis", outputCol="label")
indexed_df = indexer.fit(pca_df).transform(pca_df)

# 80% to train, 20% held back to test (seed = same split every run)
train_data, test_data = indexed_df.randomSplit([0.8, 0.2], seed=42)

lr_model = LogisticRegression(featuresCol="pca_features", labelCol="label").fit(train_data)
predictions = lr_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                              metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy * 100:.2f}% on {test_data.count()} unseen test patients")

Accuracy: 97.62% on 84 unseen test patients


## 8. Conclusion
- PCA reduced 30 measurements to 2 components that keep **63.27%** of the variance
  (PC1 = 44.22%, PC2 = 19.04%).
- **PC1** is driven mainly by tumour **concave points, concavity, compactness and perimeter**,
  i.e. the shape irregularity and size of the tumour.
- **PC2** is driven mainly by **fractal dimension** (how complex the tumour boundary is).
- These are the **essential variables** to prioritise in the donor-funding proposal.
- Bonus: logistic regression using only the 2 components predicted diagnosis with
  **97.62% accuracy** on unseen patients, so the reduced data still separates benign
  from malignant cases very well.

In [54]:
spark.stop()   # shut down the Spark engine